In [ ]:
#Install packages
system("conda install -y r-rlang=1.1.7 r-dplyr=1.1.4 r-ggplot2=3.5.1 r-tidyr=1.3.1 r-jsonLite=2.0.0 r-scales=1.4.0")

In [ ]:
#Load packages
library(dplyr)
library(ggplot2)
library(tidyr)
library(scales)
library(jsonlite)

In [ ]:
#Load parameters
params <- fromJSON("galaxy_inputs/galaxy_inputs.json")
df_path <- as.character(params$table[1])

In [ ]:
#Colour-blind-safe palette (Okabe-Ito)
CB_NE <- c(
  "Raw NeLD"               = "#E69F00",
  "Corrected (pseudorep)"  = "#56B4E9",
  "Corrected (Nb\u2192Ne)" = "#009E73"
)
## General-purpose colours — accessed by name, not tied to Ne labels
CB_ORANGE <- "#E69F00"
CB_BLUE   <- "#56B4E9"
CB_DKBLUE <- "#0072B2"
CB_RED    <- "#D55E00"

In [ ]:
#Load data and clean
raw <- read.table(df_path, header = TRUE, sep = "\t",
                  stringsAsFactors = FALSE, na.strings = c("NA", ""))

EXCLUDE_POP <- c("mean", "All_populations_combined")

pops    <- raw %>% filter(!Pop %in% EXCLUDE_POP)
sp_mean <- raw %>% filter(Pop == "mean")

## Species name extracted directly from the table
SPECIES_TAG <- unique(pops$Species_name)[1]

In [ ]:
###### Plot 1 : Ne forest plot #####

## Define which Ne estimates are available in this dataset.
ne_estimates_all <- list(
  list(label  = "Raw NeLD",
       col_ne = "NeLD",
       col_lo = "JK_CI_down",
       col_hi = "JK_CI_up"),
  list(label  = "Corrected (pseudorep)",
       col_ne = "NeLD_corrected_pseudorep",
       col_lo = "JK_CI_down_pseudorep",
       col_hi = "JK_CI_up_pseudorep"),
  list(label  = "Corrected (Nb\u2192Ne)",
       col_ne = "NeLD_corrected_Nb",
       col_lo = "JK_CI_down_Nb",
       col_hi = "JK_CI_up_Nb")
)

present_cols <- names(raw)
ne_estimates <- Filter(
  function(e) all(c(e$col_ne, e$col_lo, e$col_hi) %in% present_cols),
  ne_estimates_all
)

if (length(ne_estimates) == 0) {
  warning("No Ne estimate columns found — plot 1 will be skipped.")
  p1 <- NULL
} else {
  
  ## Build long table dynamically
  ne_long <- lapply(ne_estimates, function(e) {
    pops %>%
      transmute(
        Pop        = Pop,
        Correction = e$label,
        Ne         = .data[[e$col_ne]],
        CI_lo      = .data[[e$col_lo]],
        CI_hi      = .data[[e$col_hi]]
      )
  }) %>% bind_rows()
  
  corr_levels        <- sapply(ne_estimates, '[[', "label")
  ne_long$Correction <- factor(ne_long$Correction, levels = corr_levels)
  
  pop_levels  <- sort(unique(ne_long$Pop))
  ne_long$Pop <- factor(ne_long$Pop, levels = rev(pop_levels))
  
  ## X-axis: truncate at max finite Ne + 15 % margin (999999 clipped off)
  #Retrieve the maximum between Ne and CI_up 
  Ne_CIup_vals <- c(ne_long$Ne, ne_long$CI_hi)
  Ne_CIup_vals <- Ne_CIup_vals[Ne_CIup_vals != 999999] #remove 999999 because = INFINITE
  
  # 2. On stocke le résultat dans votre variable max
  val_max <- max(Ne_CIup_vals, na.rm = TRUE)
  val_max <- val_max*1.15 #add a margin to the maximum value
  
  #Same thing with lower margin
  Ne_CIdown_vals <- c(ne_long$Ne, ne_long$CI_lo)
  
  # 2. On stocke le résultat dans votre variable max
  val_min <- min(Ne_CIdown_vals_min, na.rm = TRUE)
  val_min <- val_min/1.15 #add a margin to the minimum value
  
  #Plot colours
  corr_colours <- CB_NE[corr_levels]
  corr_shapes  <- c(16, 17, 15)[seq_along(corr_levels)]
  
  p1 <- ggplot(data = ne_long, aes(y = Pop, x = Ne, color = Correction, group = Correction)) +
    theme_minimal() +
    geom_pointrange(
      aes(xmin = CI_lo, xmax = CI_hi, shape = Correction),
      position = position_dodge(width = 0.6),
      size = 0.8
    ) +
    scale_colour_manual(values = corr_colours, breaks = corr_levels) +
    scale_shape_manual(values  = corr_shapes,  breaks = corr_levels) +
    coord_cartesian(xlim = c(val_min, val_max)) +
    scale_y_discrete(limits = rev) + 
    labs(
      title    = "Effective population size (Ne) estimates",
      subtitle = paste0("Jackknife 95% CI  |  X-axis truncated at ", round(val_max),
                        "  |  ", SPECIES_TAG),
      x        = "Ne",
      y        = "Population"
    ) +
    theme_bw(base_size = 12) +
    theme(
      legend.position  = "bottom",
      panel.grid.minor = element_blank()
    )
}

In [ ]:
##### Plot 2 : Barplot Hobs & Hexp

h_long <- pops %>%
  select(Pop, Hobs, Hexp) %>%
  pivot_longer(cols = c(Hobs, Hexp), names_to = "Metric", values_to = "Value") %>%
  mutate(Pop = factor(Pop, levels = sort(unique(Pop))))

mean_hobs <- sp_mean$Hobs[1]
mean_hexp <- sp_mean$Hexp[1]

p2 <- ggplot(h_long, aes(x = Pop, y = Value, fill = Metric)) +
  geom_col(position = "dodge", width = 0.7, colour = "white", linewidth = 0.3) +
  geom_hline(yintercept = mean_hobs, linetype = "dashed",
             colour = CB_ORANGE, linewidth = 0.8) +
  geom_hline(yintercept = mean_hexp, linetype = "dotted",
             colour = CB_BLUE, linewidth = 0.8) +
  annotate("text", x = Inf, y = mean_hobs,
           label = paste("Species mean Hobs =", round(mean_hobs, 3)),
           hjust = 1.05, vjust = -0.4, size = 3,
           colour = CB_ORANGE) +
  annotate("text", x = Inf, y = mean_hexp,
           label = paste("Species mean Hexp =", round(mean_hexp, 3)),
           hjust = 1.05, vjust = 1.4, size = 3,
           colour = CB_BLUE) +
  scale_fill_manual(values = c(
    Hobs = CB_ORANGE,
    Hexp = CB_BLUE
  )) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.1))) +
  labs(
    title    = "Observed and expected heterozygosity",
    subtitle = SPECIES_TAG,
    x        = "Population",
    y        = "Heterozygosity",
    fill     = NULL
  ) +
  theme_bw(base_size = 12) +
  theme(
    axis.text.x        = element_text(angle = 35, hjust = 1),
    legend.position    = "top",
    panel.grid.major.x = element_blank(),
    panel.grid.minor   = element_blank()
  )


In [ ]:
##### Plot 3 : Diverging barplot Fis #####
fis_df <- pops %>%
  select(Pop, Fis) %>%
  filter(!is.na(Fis)) %>%
  mutate(
    Pop       = factor(Pop, levels = Pop[order(Fis)]),
    Direction = ifelse(Fis >= 0, "Positive", "Negative")
  )

p3 <- ggplot(fis_df, aes(x = Fis, y = Pop, fill = Direction)) +
  geom_col(width = 0.65, colour = "white", linewidth = 0.3) +
  geom_vline(xintercept = 0, linewidth = 0.8, colour = "grey20") +
  scale_fill_manual(values = c(
    Positive = CB_DKBLUE,
    Negative = CB_RED
  )) +
  scale_x_continuous(
    labels = label_number(accuracy = 0.01),
    expand = expansion(mult = c(0.05, 0.05))
  ) +
  labs(
    title    = "Inbreeding coefficient (Fis)",
    subtitle = paste("Positive = excess homozygosity  |  Negative = excess heterozygosity  |",
                     SPECIES_TAG),
    x        = "Fis",
    y        = "Population",
    fill     = NULL
  ) +
  theme_bw(base_size = 12) +
  theme(
    legend.position    = "top",
    panel.grid.major.y = element_blank(),
    panel.grid.minor   = element_blank()
  )

In [ ]:
##### Save plot #####

png(filename="outputs/Ne_forest_plot.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p1)
dev.off()

png(filename="outputs/Hobs_Hex_barplot.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p2)
dev.off()

png(filename="outputs/Fis_barplot.png",
    width = 2400,
    height = 1800,
    res = 300)
print(p3)
dev.off()